In [2]:
from pathlib import Path
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

In [3]:
DATA_DIR = Path("../data")

TRAIN_DIR = DATA_DIR / "Training_set"
VAL_DIR = DATA_DIR / "Validation_set"
TEST_DIR = DATA_DIR / "Test_set"

TRAIN_DIR

WindowsPath('../data/Training_set')

In [4]:
train_images = [
    p for p in TRAIN_DIR.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
]

train_csv = list(TRAIN_DIR.glob("*.csv"))

print("Training images:", len(train_images))
print("Training CSV:", [p.name for p in train_csv])

Training images: 509
Training CSV: ['RFMiD_2_Training_labels.csv']


In [5]:
train_labels = pd.read_csv(
    train_csv[0],
    encoding="latin1"
)

train_labels.head()

,ID,WNL,AH,AION,ARMD,BRVO,CB,CF,CL,CME,...,RT,SOFE,ST,TD,TSLN,TV,VS,HTN,IIH,Unnamed: 52
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,NaN
1,2,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,NaN
2,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,NaN
3,4,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,NaN
4,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,NaN


In [6]:
train_labels.columns.tolist()

['ID',
 'WNL',
 'AH',
 'AION',
 'ARMD',
 'BRVO',
 'CB',
 'CF',
 'CL',
 'CME',
 'CNV',
 'CRAO',
 'CRS',
 'CRVO',
 'CSR',
 'CWS',
 'CSC',
 'DN',
 'DR',
 'EDN',
 'ERM',
 'GRT',
 'HPED',
 'HR',
 'LS',
 'MCA',
 'ME',
 'MH',
 'MHL',
 'MS',
 'MYA',
 'ODC',
 'ODE',
 'ODP',
 'ON\xa0',
 'OPDM',
 'PRH',
 'RD',
 'RHL',
 'RTR',
 'RP',
 'RPEC',
 'RS',
 'RT',
 'SOFE',
 'ST',
 'TD',
 'TSLN',
 'TV',
 'VS',
 'HTN',
 'IIH',
 'Unnamed: 52']

In [7]:
train_labels.columns = train_labels.columns.str.strip()

train_labels = train_labels.loc[
    :,
    ~train_labels.columns.str.startswith("Unnamed")
]

train_labels.columns.tolist()

['ID',
 'WNL',
 'AH',
 'AION',
 'ARMD',
 'BRVO',
 'CB',
 'CF',
 'CL',
 'CME',
 'CNV',
 'CRAO',
 'CRS',
 'CRVO',
 'CSR',
 'CWS',
 'CSC',
 'DN',
 'DR',
 'EDN',
 'ERM',
 'GRT',
 'HPED',
 'HR',
 'LS',
 'MCA',
 'ME',
 'MH',
 'MHL',
 'MS',
 'MYA',
 'ODC',
 'ODE',
 'ODP',
 'ON',
 'OPDM',
 'PRH',
 'RD',
 'RHL',
 'RTR',
 'RP',
 'RPEC',
 'RS',
 'RT',
 'SOFE',
 'ST',
 'TD',
 'TSLN',
 'TV',
 'VS',
 'HTN',
 'IIH']

In [8]:
print("DR distribution:")
print(train_labels["DR"].value_counts())

print("\nWNL distribution:")
print(train_labels["WNL"].value_counts())

DR distribution:
DR
0    467
1     42
Name: count, dtype: int64

WNL distribution:
WNL
0    352
1    157
Name: count, dtype: int64


In [9]:
image_ids = {p.stem for p in train_images}
label_ids = set(train_labels["ID"].astype(str))

missing_labels = image_ids - label_ids
missing_images = label_ids - image_ids

print("Images without labels:", len(missing_labels))
print("Labels without images:", len(missing_images))

print("\nImage IDs without labels:")
print(sorted(missing_labels))

print("\nLabel IDs without images:")
print(sorted(missing_images))

Images without labels: 2
Labels without images: 2

Image IDs without labels:
['222', '79']

Label IDs without images:
['216', '89']


In [10]:
# Create a dataframe containing only images that have matching labels

train_df = train_labels[
    train_labels["ID"].astype(str).isin(image_ids)
].copy()

# Keep only the information we need for now
train_df = train_df[["ID", "WNL", "DR"]]

# Add the corresponding image path
train_df["image_path"] = train_df["ID"].astype(str).apply(
    lambda x: next(
        p for p in train_images
        if p.stem == x
    )
)

train_df.head()

,ID,WNL,DR,image_path
0,1,0,0,..\data\Training_set\1.jpg
1,2,0,0,..\data\Training_set\2.jpg
2,3,0,0,..\data\Training_set\3.jpg
3,4,0,0,..\data\Training_set\4.jpg
4,5,0,0,..\data\Training_set\5.jpg


In [11]:
# Cross-tabulation of DR and WNL
pd.crosstab(
    train_df["DR"],
    train_df["WNL"],
    margins=True
)

WNL,0,1,All
DR,,,
0,309,156,465
1,42,0,42
All,351,156,507
